# Energy H Long CLP / MCTS Experiments

This notebook runs a compact CLP-style experiment on `electricity_H_long`, the hourly Energy config from GIFT-Eval.

The flow is:

1. Clone [`chahineNejm/kernels_playground`](https://github.com/chahineNejm/kernels_playground) and [`chahineNejm/graph_Time_series`](https://github.com/chahineNejm/graph_Time_series) in Colab.
2. Load a small slice of `electricity_H_long` using `kernels_playground/first_tests/utils`.
3. Convert the examples into arrays `H` and `F` for the graph CLP state.
4. Build the `graph_Time_series` grammar.
5. Run MCTS over cleaning -> feature -> model -> STOP chains.
6. Inspect the best chain, MCTS tree, and a graph/GIEN-style summary of what the search learned.

Default settings are intentionally small so the notebook can run interactively. Increase `N_STOP`, decrease `STEP`, raise `HISTORY_LEN`, or enable tree models once the loop is behaving.

## 0. Setup

This notebook is meant to run on Colab. The first code cell clones both GitHub repositories into `/content`, installs the kernel playground requirements, and adds the right package paths to `sys.path`.

In [ ]:
from pathlib import Path
import os
import sys
import importlib.util

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()

if IN_COLAB:
    %cd /content
    import subprocess
    import time

    def run_retry(cmd, attempts=3, delay=5):
        """Run a shell command with simple retry logic for Colab network hiccups."""
        last_exc = None
        for attempt in range(1, attempts + 1):
            print(f"[{attempt}/{attempts}]", " ".join(cmd))
            try:
                subprocess.check_call(cmd)
                return
            except subprocess.CalledProcessError as exc:
                last_exc = exc
                if attempt < attempts:
                    print(f"command failed; retrying in {delay}s...")
                    time.sleep(delay)
        raise last_exc

    def clone_or_update(repo_url, target):
        target = Path(target)
        if (target / ".git").exists():
            run_retry(["git", "-C", str(target), "pull", "--ff-only"])
        else:
            run_retry(["git", "clone", "--depth", "1", repo_url, str(target)])

    clone_or_update("https://github.com/chahineNejm/kernels_playground.git", "/content/kernels_playground")
    clone_or_update("https://github.com/chahineNejm/graph_Time_series.git", "/content/graph_Time_series")

    run_retry([
        sys.executable, "-m", "pip", "install", "-q",
        "-r", "/content/kernels_playground/first_tests/requirements.txt",
        "scikit-learn", "networkx", "tqdm"
    ])

    KERNELS_REPO = Path("/content/kernels_playground")
    GRAPH_REPO = Path("/content/graph_Time_series")
else:
    # Local fallback for running this notebook from the graph construction folder.
    GRAPH_REPO = Path.cwd() / "graph_Time_series"
    KERNELS_REPO = Path.cwd().parent / "kernels_playground"

FIRST_TESTS = KERNELS_REPO / "first_tests"

# Import layout notes:
# - kernels_playground exposes utils from first_tests/utils, so add first_tests.
# - graph_Time_series may be a package directory itself; adding /content lets
#   `import graph_Time_series` work when cloned to /content/graph_Time_series.
# - adding GRAPH_REPO too also supports repo layouts with a nested package.
for p in [FIRST_TESTS, GRAPH_REPO.parent, GRAPH_REPO]:
    p = str(p.resolve())
    if p not in sys.path:
        sys.path.insert(0, p)

print("IN_COLAB:", IN_COLAB)
print("KERNELS_REPO:", KERNELS_REPO, KERNELS_REPO.exists())
print("FIRST_TESTS:", FIRST_TESTS, FIRST_TESTS.exists())
print("GRAPH_REPO:", GRAPH_REPO, GRAPH_REPO.exists())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint

from utils.config import DATASETS
from utils.data import build_examples
from utils.augmentation import uniform_length

from graph_Time_series import State, Grammar, plot_grammar, mcts_search, print_mcts_tree
from graph_Time_series.tokens.cleaning import (
    CleanIdentity,
    CleanDetrend,
    CleanMovingAvg,
    CleanNormalize,
    CleanDetrendNorm,
)
from graph_Time_series.tokens.features import FeatRaw, FeatFFTEncode, FeatLagFeatures
from graph_Time_series.tokens.models import ModelKernelRBF, ModelRandomForest, ModelXGBoost, StopToken, compute_mase

np.set_printoptions(precision=4, suppress=True)

## 1. Experiment Knobs

`electricity_H_long` can be large, and the graph MCTS evaluates pipelines with leave-one-out model predictions. Start small, then scale up.

Notes:

- `HISTORY_LEN=512` keeps model inputs manageable.
- `FUTURE_LEN=96` evaluates the first 96 forecast steps, not the whole long horizon.
- `ENABLE_TREE_MODELS=False` makes the default run fast by using only the kernel model token.
- Turn on tree models after the RBF-only search works.

In [ ]:
CONFIG_NAME = "electricity_H_long"  # Energy, hourly, long horizon

# Data slice. With start=0, stop=120, step=4 this gives about 30 samples.
N_START = 0
N_STOP = 120
STEP = 4

# Shape controls for CLP/MCTS.
HISTORY_LEN = 512
FUTURE_LEN = 96

# MCTS controls.
N_MCTS_ITERATIONS = 25
PUCT_C = 1.5

# Keep False for a quick first run. True adds explicit LOO RF/XGB models.
ENABLE_TREE_MODELS = False
ENABLE_XGBOOST = False

SEED = 0

## 2. Load `electricity_H_long`

This uses `build_examples` and `uniform_length` from the existing kernel playground. Only histories are resized; futures are cropped to `FUTURE_LEN` in this notebook.

In [ ]:
raw = build_examples(
    config=CONFIG_NAME,
    start=N_START,
    stop=N_STOP,
    step=STEP,
    dataset_name=DATASETS["eval"],
)

fixed = uniform_length(
    raw,
    target_len=HISTORY_LEN,
    min_len=HISTORY_LEN // 2,
    keys=("history",),
    seed=SEED,
    verbose=True,
)

# Keep examples with enough future horizon.
fixed = [r for r in fixed if len(r["future"]) >= FUTURE_LEN]

H = np.stack([np.asarray(r["history"], dtype=np.float32) for r in fixed])
F = np.stack([np.asarray(r["future"][:FUTURE_LEN], dtype=np.float32) for r in fixed])

print(f"loaded raw examples: {len(raw)}")
print(f"usable fixed examples: {len(fixed)}")
print("H shape:", H.shape)
print("F shape:", F.shape)
assert H.ndim == 2 and F.ndim == 2
assert H.shape[0] == F.shape[0]
assert H.shape[0] >= 6, "MCTS/LOO needs a few samples; increase N_STOP or reduce STEP."

In [ ]:
def plot_example_grid(H, F, n=6):
    n = min(n, H.shape[0])
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(14, 3.2 * rows), squeeze=False)
    for i in range(rows * cols):
        ax = axes[i // cols, i % cols]
        if i >= n:
            ax.set_visible(False)
            continue
        h = H[i]
        f = F[i]
        ax.plot(np.arange(-len(h), 0), h, color="0.55", lw=1.0, label="history")
        ax.plot(np.arange(len(f)), f, color="black", lw=1.2, label="future")
        ax.axvline(0, color="tab:blue", ls=":", lw=0.8)
        ax.set_title(f"sample {i}")
        ax.grid(True, alpha=0.2)
        if i == 0:
            ax.legend(fontsize=8)
    fig.tight_layout()
    return fig

plot_example_grid(H, F, n=6);

## 3. Build the CLP Grammar

This grammar is the computational language. Tokens are small operations, and edges say which operations may follow which.

The default grammar searches over:

- cleaning: identity, detrend, moving average, normalize, detrend+normalize
- features: raw histories, FFT summary features, lag/autocorrelation features
- model: RBF kernel regression
- control: STOP

Tree models are optional because they use explicit leave-one-out loops and can be slower.

In [ ]:
def module_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def build_energy_grammar(enable_tree_models=False, enable_xgboost=False):
    grammar = Grammar()

    # Cleaning tokens.
    cleaning_tokens = [
        CleanIdentity(),
        CleanDetrend(),
        CleanMovingAvg(),
        CleanNormalize(),
        CleanDetrendNorm(),
    ]
    for tok in cleaning_tokens:
        grammar.register(tok, follows=["START"])

    # Feature tokens.
    feature_tokens = [FeatRaw(), FeatFFTEncode(), FeatLagFeatures()]
    cleaning_names = [t.name for t in cleaning_tokens]
    for prev in cleaning_names:
        for tok in feature_tokens:
            grammar.register(tok, follows=[prev])

    # Model tokens. Kernel is always available. Tree models are optional.
    model_tokens = [ModelKernelRBF()]
    if enable_tree_models:
        model_tokens.append(ModelRandomForest())
    if enable_xgboost and module_available("xgboost"):
        model_tokens.append(ModelXGBoost())
    elif enable_xgboost:
        print("xgboost not installed; skipping xgboost token.")

    feature_names = [t.name for t in feature_tokens]
    model_names = [t.name for t in model_tokens]
    model_follows = feature_names + model_names
    model_leads = model_names + ["STOP"]
    for tok in model_tokens:
        grammar.register(tok, follows=model_follows, leads_to=model_leads)

    grammar.register(StopToken(), follows=[])
    return grammar


grammar = build_energy_grammar(
    enable_tree_models=ENABLE_TREE_MODELS,
    enable_xgboost=ENABLE_XGBOOST,
)
grammar

In [ ]:
print("tokens:", grammar.token_names)
print("edges:", grammar.graph.number_of_edges())
plot_grammar(grammar, title="CLP grammar for electricity_H_long")

## 4. Baseline Sanity Checks

Before running MCTS, manually evaluate a few chains. This helps separate data problems from search problems.

In [ ]:
def run_chain(H, F, token_names, grammar):
    state = State(H, F)
    for name in token_names:
        state = grammar.tokens[name].apply(state)
    if not state.terminated:
        state = grammar.tokens["STOP"].apply(state)
    return state


baseline_chains = [
    ["identity", "feat_raw", "kernel_rbf", "STOP"],
    ["normalize", "feat_raw", "kernel_rbf", "STOP"],
    ["normalize", "feat_lag", "kernel_rbf", "STOP"],
    ["detrend_norm", "feat_lag", "kernel_rbf", "STOP"],
    ["moving_avg", "fft_encode", "kernel_rbf", "STOP"],
]

baseline_results = []
for chain in baseline_chains:
    s = run_chain(H, F, chain, grammar)
    baseline_results.append({"chain": " -> ".join(chain), "mase": s.mase})

baseline_results = sorted(baseline_results, key=lambda x: x["mase"])
for row in baseline_results:
    print(f"MASE={row['mase']:.4f} | {row['chain']}")

## 5. Run MCTS

MCTS rolls out token chains, scores them with MASE, and backs up the reward through the tree. Lower MASE is better; the reward is `1 / (1 + MASE)`.

In [ ]:
initial_state = State(H, F)

mcts_results = mcts_search(
    grammar,
    initial_state,
    n_iterations=N_MCTS_ITERATIONS,
    puct_c=PUCT_C,
    verbose=True,
)

print("\nBest chain:", mcts_results["best_chain"])
print("Best MASE:", mcts_results["best_mase"])

In [ ]:
print_mcts_tree(mcts_results["root"], max_depth=6)

## 6. MCTS History and Best Chains

This section turns the search trace into simple tables and plots. It avoids a pandas dependency so it works in lighter environments.

In [ ]:
history = mcts_results["history"]
mase_curve = np.array([h["mase"] for h in history], dtype=float)
best_so_far = np.minimum.accumulate(mase_curve)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(mase_curve, marker="o", lw=1, label="iteration MASE")
ax.plot(best_so_far, marker="s", lw=2, label="best so far")
ax.set_xlabel("MCTS iteration")
ax.set_ylabel("MASE")
ax.set_title("MCTS search progress")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout();

In [ ]:
top_chains = sorted(mcts_results["all_chains"].items(), key=lambda kv: kv[1])[:10]
print("Top chains")
print("-" * 100)
for chain, mase in top_chains:
    print(f"MASE={mase:8.4f} | {chain}")

## 7. Re-run the Best Chain and Inspect Forecasts

MCTS gives us the best token sequence it found. Replaying it gives a final `State` with the forecast, transformation log, and residual stack.

In [ ]:
best_tokens = [t.strip() for t in mcts_results["best_chain"].split("->")]
best_state = run_chain(H, F, best_tokens, grammar)

print("best tokens:", best_tokens)
print("best MASE:", best_state.mase)
best_state.print_log()

In [ ]:
forecast = best_state.features["final_forecast"]

def plot_forecasts(H, F, forecast, n=6):
    n = min(n, H.shape[0])
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(14, 3.4 * rows), squeeze=False)
    for i in range(rows * cols):
        ax = axes[i // cols, i % cols]
        if i >= n:
            ax.set_visible(False)
            continue
        h_tail = H[i, -min(160, H.shape[1]):]
        ax.plot(np.arange(-len(h_tail), 0), h_tail, color="0.65", lw=1.0, label="history")
        ax.plot(np.arange(F.shape[1]), F[i], color="black", lw=1.4, label="actual")
        ax.plot(np.arange(forecast.shape[1]), forecast[i], color="tab:orange", ls="--", lw=1.4, label="forecast")
        ax.axvline(0, color="tab:blue", ls=":", lw=0.8)
        ax.set_title(f"sample {i}")
        ax.grid(True, alpha=0.2)
        if i == 0:
            ax.legend(fontsize=8)
    fig.tight_layout()
    return fig

plot_forecasts(H, F, forecast, n=6);

## 8. GIEN-Style Graph Interpretation

Here GIEN means a graph interpretation / experiment-notebook view: instead of only reporting the winning chain, inspect which tokens the search visited and which grammar branches looked useful.

In [ ]:
def flatten_tree(node, rows=None, prefix=()):
    if rows is None:
        rows = []
    for child in node.children.values():
        path = prefix + (child.name,)
        est_mase = (1.0 / child.avg_reward - 1.0) if child.avg_reward > 0 else float("inf")
        rows.append({
            "path": " -> ".join(path),
            "token": child.name,
            "visits": child.visits,
            "avg_reward": child.avg_reward,
            "est_mase": est_mase,
            "prior": child.prior,
        })
        flatten_tree(child, rows, path)
    return rows


tree_rows = flatten_tree(mcts_results["root"])
tree_rows_sorted = sorted(tree_rows, key=lambda r: (-r["visits"], r["est_mase"]))

print("Most visited tree nodes")
print("-" * 120)
for r in tree_rows_sorted[:20]:
    print(f"visits={r['visits']:3d} est_MASE={r['est_mase']:8.4f} prior={r['prior']:.3f} | {r['path']}")

In [ ]:
token_visits = {}
for r in tree_rows:
    token_visits[r["token"]] = token_visits.get(r["token"], 0) + r["visits"]

items = sorted(token_visits.items(), key=lambda kv: kv[1], reverse=True)
labels = [k for k, _ in items]
values = [v for _, v in items]

fig, ax = plt.subplots(figsize=(10, max(3, 0.35 * len(labels))))
ax.barh(labels[::-1], values[::-1], color="tab:blue", alpha=0.75)
ax.set_xlabel("visits across MCTS tree")
ax.set_title("Token visit mass")
ax.grid(True, axis="x", alpha=0.25)
fig.tight_layout();

## 9. Next Experiment Ideas

Good next moves after the first run:

- Increase `N_MCTS_ITERATIONS` to 100+ once the search is stable.
- Increase `N_STOP` or reduce `STEP` to give the LOO models more examples.
- Try `HISTORY_LEN` values like 256, 512, 1024, and 2048.
- Turn on `ENABLE_TREE_MODELS=True` to let MCTS compare RBF with random forest.
- Turn on `ENABLE_XGBOOST=True` if `xgboost` is installed.
- Add a new token for the masked RBF idea from `first_tests/utils/kernels.py`.
- Add decomposition tokens for CWT/KMD/EMD if the feature graph starts to plateau.